Author: Malachy McCaffrey
Date: 06/19/2026


Space time analysis workflow for GFW fishing vessel presence hours data


Scenarios:
1. Grid size: 1km, 2km
2. Metrics:
   - SUM hours (raw values and log-transformed post-STC)
   - COUNT records
   - UNIQUE Vessel IDs
3. Analyses:
   - Emerging Hot Spot Analysis
   - Mann-Kendall Trend Analysis
   - Local Outlier Analysis

Call ArcPy and set working environment

In [1]:
import arcpy
from arcpy import env
import os
import csv

env.workspace = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb"
env.overwriteOutput = True

workspace = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb"

Import point feature class

In [6]:
gfw_vp_fv_stc = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_utm19n"

# confirm spatial ref

desc = arcpy.Describe(gfw_vp_fv_stc)
print(desc.spatialReference.name)

# fields


date = "Year_Month"
ref_time = "1/1/2016"

WGS_1984_UTM_Zone_19N


Define output paths

In [3]:
cube_1km = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\gfw_vp_fv_ptAgg1km.nc"

cube_2km = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\gfw_vp_fv_ptAgg2km.nc"

stc_comparison = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\gfw_vp_fv_stcComparison.csv"

Create 1km STC

In [4]:
help(arcpy.stpm.CreateSpaceTimeCube)

Help on function CreateSpaceTimeCube in module arcpy.stpm:

CreateSpaceTimeCube(
    in_features=None,
    output_cube=None,
    time_field=None,
    template_cube=None,
    time_step_interval=None,
    time_step_alignment: "Literal['END_TIME', 'START_TIME', 'REFERENCE_TIME'] | None" = None,
    reference_time=None,
    distance_interval=None,
    summary_fields=None,
    aggregation_shape_type: "Literal['FISHNET_GRID', 'HEXAGON_GRID', 'DEFINED_LOCATIONS'] | None" = None,
    defined_polygon_locations=None,
    location_id=None
) -> 'Result1[str | Path]'
    CreateSpaceTimeCube_stpm(in_features, output_cube, time_field, {template_cube}, {time_step_interval}, {time_step_alignment}, {reference_time}, {distance_interval}, {summary_fields;summary_fields...}, {aggregation_shape_type}, {defined_polygon_locations}, {location_id})

       Summarizes a set of points into a netCDF data structure by aggregating
       them into space-time bins.

    INPUTS:
     in_features (Feature Layer):
     

In [15]:
print("Creating 1km Cube")

arcpy.stpm.CreateSpaceTimeCube(
    in_features=gfw_vp_fv_stc,
    output_cube=cube_1km,
    time_field=date,
    time_step_interval="1 Months",
    time_step_alignment="REFERENCE_TIME",
    reference_time=ref_time,
    aggregation_shape_type="FISHNET_GRID",
    distance_interval="1000 Meters",
    summary_fields=[
        ["Vessel_Presence_Hours", "SUM", "ZEROS"], # missed observations assigned zeros
        ["Vessel_Count", "SUM", "ZEROS"]
    ]
)

print("1km Cube Complete")

print(arcpy.GetMessages())

Creating 1km Cube
1km Cube Complete
Start Time: Friday, July 10, 2026 11:56:34 AM
The space time cube has aggregated 65948 points into 4071 fishnet grid locations over 125 time step intervals.  Each location is a 1000 meters by 1000 meters square.  The entire space time cube spans an area 59000 meters west to east and 69000 meters north to south.  Each of the time step intervals is 1 month in duration so the entire time period covered by the space time cube is 125 months.  Of the 4071 total locations, 2838 (69.71%) contain at least one point for at least one time step interval.  These 2838 locations comprise 354750 space time bins of which 61955 (17.46%) have point counts greater than zero.  There is a statistically significant increase in point counts over time.

---------- Space Time Cube Characteristics -----------
Input feature time extent          2016-01-01 00:00:00
                                to 2026-05-01 00:00:00
                                                      
Numbe

Create 2km STC

In [16]:
print("Creating 2km Cube")

arcpy.stpm.CreateSpaceTimeCube(
    in_features=gfw_vp_fv_stc,
    output_cube=cube_2km,
    time_field=date,
    time_step_interval="1 Months",
    time_step_alignment="REFERENCE_TIME",
    reference_time=ref_time,
    aggregation_shape_type="FISHNET_GRID",
    distance_interval="2000 Meters",
    summary_fields=[
        ["Vessel_Presence_Hours", "SUM", "ZEROS"], # missed observations assigned zeros
        ["Vessel_Count", "SUM", "ZEROS"]
    ]
)

print("2km Cube Complete")

print(arcpy.GetMessages())

# OCCUPANY RATE AND PERCENT NON-ZERO BINS

Creating 2km Cube
2km Cube Complete
Start Time: Friday, July 10, 2026 11:56:41 AM
The space time cube has aggregated 65948 points into 1050 fishnet grid locations over 125 time step intervals.  Each location is a 2000 meters by 2000 meters square.  The entire space time cube spans an area 60000 meters west to east and 70000 meters north to south.  Each of the time step intervals is 1 month in duration so the entire time period covered by the space time cube is 125 months.  Of the 1050 total locations, 818 (77.90%) contain at least one point for at least one time step interval.  These 818 locations comprise 102250 space time bins of which 38757 (37.90%) have point counts greater than zero.  There is a statistically significant increase in point counts over time.

---------- Space Time Cube Characteristics -----------
Input feature time extent          2016-01-01 00:00:00
                                to 2026-05-01 00:00:00
                                                      
Number 

Create 2D trend surfaces of hours and vessels variables for both STCs.

In [13]:
help(arcpy.stpm.VisualizeSpaceTimeCube2D)

Help on function VisualizeSpaceTimeCube2D in module arcpy.stpm:

VisualizeSpaceTimeCube2D(
    in_cube=None,
    cube_variable=None,
    display_theme: "Literal['LOCATIONS_WITH_DATA', 'TRENDS', 'HOT_AND_COLD_SPOT_TRENDS', 'EMERGING_HOT_SPOT_ANALYSIS_RESULTS', 'LOCAL_OUTLIER_ANALYSIS_RESULTS', 'PERCENTAGE_OF_LOCAL_OUTLIERS', 'LOCAL_OUTLIER_IN_MOST_RECENT_TIME_PERIOD', 'TIME_SERIES_CLUSTERING_RESULTS', 'LOCATIONS_WITHOUT_SPATIAL_NEIGHBORS', 'NUMBER_OF_ESTIMATED_BINS', 'LOCATIONS_EXCLUDED_FROM_ANALYSIS', 'FORECAST_RESULTS', 'TIME_SERIES_OUTLIER_RESULTS', 'TIME_SERIES_CHANGE_POINTS', 'TIME_SERIES_CROSS_CORRELATION_RESULTS', 'FOOTPRINTS'] | None" = None,
    output_features=None,
    enable_time_series_popups: "Literal['CREATE_POPUP', 'NO_POPUP'] | None" = None
) -> 'Result1[str | Path]'
    VisualizeSpaceTimeCube2D_stpm(in_cube, cube_variable, display_theme, output_features, {enable_time_series_popups})

       Visualizes the variables stored in a netCDF space-time cube and the
       resu

In [19]:
# 1km vessel presence hours

hours_trend_1km = (
    r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_1km_hours_trend"
)

arcpy.stpm.VisualizeSpaceTimeCube2D(
    in_cube=cube_1km,
    cube_variable="VESSEL_PRESENCE_HOURS_SUM_ZEROS",
    display_theme="TRENDS",
    output_features=hours_trend_1km
)

# streaks of missing grid cells a result of STC misalignment with original grid. Make 1km grid cell with defined locations tool.

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_1km_hours_trend'>

In [21]:
# 1km vessel counts

vessels_trend_1km = (
    r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_1km_vessels_trend"
)

arcpy.stpm.VisualizeSpaceTimeCube2D(
    in_cube=cube_1km,
    cube_variable="VESSEL_COUNT_SUM_ZEROS",
    display_theme="TRENDS",
    output_features=vessels_trend_1km
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_1km_vessels_trend'>

In [22]:
# 2km vessel presence hours

hours_trend_2km = (
    r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_2km_hours_trend"
)

arcpy.stpm.VisualizeSpaceTimeCube2D(
    in_cube=cube_2km,
    cube_variable="VESSEL_PRESENCE_HOURS_SUM_ZEROS",
    display_theme="TRENDS",
    output_features=hours_trend_2km
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_2km_hours_trend'>

In [23]:
# 2km vessel counts

vessels_trend_2km = (
    r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_2km_vessels_trend"
)

arcpy.stpm.VisualizeSpaceTimeCube2D(
    in_cube=cube_2km,
    cube_variable="VESSEL_COUNT_SUM_ZEROS",
    display_theme="TRENDS",
    output_features=vessels_trend_2km
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_2km_vessels_trend'>

1km grid cell with STC defined locations

In [25]:
# input polygon fc

stcGrid = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_stcGrid_utm19n"

# output path

polyCube = r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\gfw_vp_fv_defLoc1km.nc"

In [28]:
help(arcpy.stpm.CreateSpaceTimeCubeDefinedLocations)

Help on function CreateSpaceTimeCubeDefinedLocations in module arcpy.stpm:

CreateSpaceTimeCubeDefinedLocations(
    in_features=None,
    output_cube=None,
    location_id=None,
    temporal_aggregation: "Literal['APPLY_TEMPORAL_AGGREGATION', 'NO_TEMPORAL_AGGREGATION'] | None" = None,
    time_field=None,
    time_step_interval=None,
    time_step_alignment: "Literal['END_TIME', 'START_TIME', 'REFERENCE_TIME'] | None" = None,
    reference_time=None,
    variables=None,
    summary_fields=None,
    in_related_table=None,
    related_location_id=None
) -> 'Result1[str | Path]'
    CreateSpaceTimeCubeDefinedLocations_stpm(in_features, output_cube, location_id, temporal_aggregation, time_field, time_step_interval, {time_step_alignment}, {reference_time}, {variables;variables...}, {summary_fields;summary_fields...}, {in_related_table}, {related_location_id})

       Structures panel data or station data (defined locations where
       geography does not change but attributes are changing 

In [35]:
print("Creating 1km Cube with Defined Locations")

arcpy.stpm.CreateSpaceTimeCubeDefinedLocations(
    in_features=stcGrid,
    output_cube=polyCube,
    location_id= "Grid_ID",
    temporal_aggregation= "APPLY_TEMPORAL_AGGREGATION",
    time_field=date,
    time_step_interval="1 Months",
    time_step_alignment="REFERENCE_TIME",
    reference_time=ref_time,
    summary_fields= [
        ["Vessel_Presence_Hours", "SUM", "ZEROS"],
        ["Vessel_Count", "SUM", "ZEROS"]
    ]
)

print("1km Cube with Defined Locations Complete")

print(arcpy.GetMessages())

Creating 1km Cube with Defined Locations
1km Cube with Defined Locations Complete
Start Time: Friday, July 10, 2026 1:47:48 PM

---------- Space Time Cube Characteristics -----------
Input feature time extent          2016-01-01 00:00:00
                                to 2026-05-01 00:00:00
                                                      
Number of time steps                               125
Time step interval                             1 month
Time step alignment                              Start
                                                      
First time step temporal bias                    0.00%
First time step interval                   on or after
                                   2016-01-01 00:00:00
                                             to before
                                   2016-02-01 00:00:00
                                                      
Last time step temporal bias                   100.00%
Last time step interval                    on o

In [36]:
# 2D vessel presence hours trend

defLoc_hours_trend_1km = (
    r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_defLoc_1km_hours_trend"
)

arcpy.stpm.VisualizeSpaceTimeCube2D(
    in_cube=polyCube,
    cube_variable="VESSEL_PRESENCE_HOURS_SUM_ZEROS",
    display_theme="TRENDS",
    output_features=defLoc_hours_trend_1km
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_defLoc_1km_hours_trend'>

In [37]:
# 2D vessel counts trend

defLoc_vessels_trend_1km = (
    r"C:\\Users\\mmccaffrey17\\ArcGIS\\Projects\\VesselResponses_SNE\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_defLoc_1km_vessels_trend"
)

arcpy.stpm.VisualizeSpaceTimeCube2D(
    in_cube=polyCube,
    cube_variable="VESSEL_COUNT_SUM_ZEROS",
    display_theme="TRENDS",
    output_features=defLoc_vessels_trend_1km
)

<Result 'C:\\\\Users\\\\mmccaffrey17\\\\ArcGIS\\\\Projects\\\\VesselResponses_SNE\\\\VesselResponses_SNE.gdb\\gfw_vp_fv_stc_defLoc_1km_vessels_trend'>

In [ ]:
# create STC layer


help(arcpy.stpm.MakeSpaceTimeCubeLayer)

Help on function MakeSpaceTimeCubeLayer in module arcpy.stpm:

MakeSpaceTimeCubeLayer(
    in_cube=None,
    output_features=None,
    cube_variables=None,
    out_geometry_type: "Literal['POINT', 'POLYGON'] | None" = None
) -> 'Result1[str | Path]'
    MakeSpaceTimeCubeLayer_stpm(in_cube, output_features, {cube_variables;cube_variables...}, {out_geometry_type})

       Creates a space-time cube layer from a netCDF space-time cube that was
       created using a tool from the Space Time Cube Creation toolset.

    INPUTS:
     in_cube (File):
         The space-time cube that will be used as the source for the output
         space-time cube layer.A space-time cube has an .nc file extension and
         was created by a tool
         in the Space Time Cube Creation toolset.
     cube_variables {String}:
         The variables from the input space-time cube that will be included in
         the output space-time cube layer. By default, all variables will be
         included.
     out_g

need to create 2km fishnet and aggregate data by that then use create stc by defined locations